In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import lamindb as ln
import squidpy as sq
from scipy.stats import binom, norm, poisson, median_abs_deviation
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

ln.track("mzwkkCrWFfJ9")

In [ ]:
FIGURES_DIR = '/home/workspace/PCD_data/xenium/commercial/figures/qc_filtering_gmm_enrichment'

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
def save_fig(fig, stem, description, ulabels=(), features=None):
    for ax in fig.axes:
        for coll in ax.collections:
            coll.set_rasterized(True)
    for ext in ['png', 'pdf']:
        path = os.path.join(FIGURES_DIR, f'{stem}.{ext}')
        fig.savefig(path, dpi=150, bbox_inches='tight')
        artifact = ln.Artifact(path, key=f'{stem}.{ext}', description=description).save()
        if ulabels:
            artifact.ulabels.add(*ulabels)
        if features:
            artifact.features.add_values(features)
    plt.close(fig)

In [ ]:
SAMPLE_NAMES = [
    'TIS05393',
    'TIS04788',
    'TIS05392',
    'TIS05394',
    'TIS04789',
    'TIS05397_008',
]

QC_COLUMNS = ['total_counts', 'n_genes_by_counts']

METRICS_PATH = '/home/workspace/PCD_data/xenium/commercial/data/metrics_summary_all.csv'

NOISE_QUANTILE = 0.999
NMADS          = 5

In [ ]:
samples = {}
qc      = {}
spatial = {}
for name in SAMPLE_NAMES:
    # Loads the raw per-sample AnnData from lamindb storage.
    adata = ln.Artifact.get(key=f'{name}.h5ad').load()
    # Adds obs['n_genes_by_counts'], obs['log1p_n_genes_by_counts'] and obs['log1p_total_counts'], and overwrites
    # obs['total_counts'] with the row sum of .X. .X is not changed; in memory only.
    sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
    samples[name] = adata
    qc[name] = adata.obs[['total_counts', 'n_genes_by_counts', 'cell_area']].rename(columns={'n_genes_by_counts': 'n_genes'})
    spatial[name] = adata.obsm['spatial']

In [ ]:
fig, axes = plt.subplots(
    len(samples), len(QC_COLUMNS),
    figsize=(5.5 * len(QC_COLUMNS), 5 * len(samples)),
    squeeze=False,
)

for i, (name, adata) in enumerate(samples.items()):
    for j, column in enumerate(QC_COLUMNS):
        ax = axes[i, j]
        # Plotting a numeric column adds no uns color entries; nothing is added to adata.
        sc.pl.spatial(
            adata, color=column, vmax='p99', spot_size=50,
            ax=ax, show=False, colorbar_loc=None,
            title=f'{name} — {column}\n(median {adata.obs[column].median():.0f})',
        )
        fig.colorbar(
            ax.collections[0],
            cax=ax.inset_axes([1.03, 0, 0.03, 1]),
        )

fig.tight_layout()
plt.show()

In [ ]:
nrows, ncols = 2, 3
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = axes.flatten()

for ax, (name, adata) in zip(axes, samples.items()):
    sc.pl.highest_expr_genes(adata, n_top=10, ax=ax, show=False)
    ax.set_title(name)

for ax in axes[len(samples):]:
    ax.set_visible(False)

fig.tight_layout()
plt.show()

## Cutoffs 1: noise floor

`estimated_number_of_false_positive_transcripts_per_cell` in the Xenium metrics is the expected number of
counts per cell from background. A cell whose counts are within the upper tail of a Poisson with that mean
cannot be told apart from noise.

In [ ]:
metrics = pd.read_csv(METRICS_PATH).set_index('sample')
noise_rate = metrics.loc[SAMPLE_NAMES, 'estimated_number_of_false_positive_transcripts_per_cell']
noise_min_counts = pd.Series(poisson.ppf(NOISE_QUANTILE, noise_rate) + 1, index=SAMPLE_NAMES)

pd.DataFrame({'false_positives_per_cell': noise_rate, 'noise_min_counts': noise_min_counts})

## Cutoffs 2: Gaussian mixture

Fitted on cells with at least one count (zero-count cells are removed by any cutoff and would add a spike at log10(0)).
`bic_gain` is BIC(one component) minus BIC(two components). With around 100k cells it is large almost whatever the shape,
so it does not show that two real humps exist; judge that from the histograms below.

In [ ]:
mixtures     = {}
mixture_rows = []
for name in SAMPLE_NAMES:
    for metric in ['total_counts', 'n_genes']:
        values     = qc[name][metric].to_numpy()
        log_values = np.log10(values[values > 0]).reshape(-1, 1)

        one_component  = GaussianMixture(1, random_state=0).fit(log_values)
        two_components = GaussianMixture(2, random_state=0).fit(log_values)

        low  = np.argmin(two_components.means_.ravel())
        high = 1 - low
        grid = np.linspace(two_components.means_.min(), two_components.means_.max(), 2000).reshape(-1, 1)
        posterior_low = two_components.predict_proba(grid)[:, low]
        cut_log10     = grid[np.argmax(posterior_low < 0.5), 0]

        mixtures[(name, metric)] = two_components
        mixture_rows.append({
            'sample': name,
            'metric': metric,
            'fraction_low': two_components.weights_[low],
            'mean_low': 10 ** two_components.means_[low, 0],
            'mean_high': 10 ** two_components.means_[high, 0],
            'min_value': np.ceil(10 ** cut_log10),
            'bic_gain': one_component.bic(log_values) - two_components.bic(log_values),
        })

mixture_table = pd.DataFrame(mixture_rows)
mixture_table

## Proposed cutoffs

Counts: the larger of the noise floor and the mixture cut. Genes: the mixture cut. Cells are kept when they are at or above both.
The fraction removed is shown against the previous fixed cutoffs of 15 counts and 10 genes.

In [ ]:
PREVIOUS_MIN_COUNTS = 15
PREVIOUS_MIN_GENES  = 10

mixture_wide = mixture_table.pivot(index='sample', columns='metric')

cutoffs = pd.DataFrame(index=SAMPLE_NAMES)
cutoffs['noise_min_counts']    = noise_min_counts
cutoffs['gmm_min_counts']      = mixture_wide[('min_value', 'total_counts')]
cutoffs['gmm_min_genes']       = mixture_wide[('min_value', 'n_genes')]
cutoffs['proposed_min_counts'] = cutoffs[['noise_min_counts', 'gmm_min_counts']].max(axis=1)
cutoffs['proposed_min_genes']  = cutoffs['gmm_min_genes']

for name in SAMPLE_NAMES:
    counts = qc[name]['total_counts']
    genes  = qc[name]['n_genes']
    previous_removed = (counts < PREVIOUS_MIN_COUNTS) | (genes < PREVIOUS_MIN_GENES)
    proposed_removed = (counts < cutoffs.loc[name, 'proposed_min_counts']) | (genes < cutoffs.loc[name, 'proposed_min_genes'])
    cutoffs.loc[name, 'median_counts']              = counts.median()
    cutoffs.loc[name, 'fraction_removed_previous']  = previous_removed.mean()
    cutoffs.loc[name, 'fraction_removed_proposed']  = proposed_removed.mean()

cutoffs.round(3)

## Cutoffs 3: sparse-tissue enrichment

Counts are grouped in steps of 5. For each step, the cells it holds are tested against the 10% of lonely cells expected by chance
(one-sided binomial test). The cutoff is the start of the first step with at least 30 cells that is not significantly lonelier
than chance, and never below the noise floor. NaN means no step up to 60 counts was quiet. This does not change `proposed_min_counts`.

In [ ]:
CANDIDATE_MIN_COUNTS = np.arange(1, 61)
NEIGHBORS            = 10
SPARSE_FRACTION      = 0.10

STEP_WIDTH         = 5
MIN_CELLS_PER_STEP = 30
ALPHA              = 0.05

first_quiet_step_start = {}
for name in SAMPLE_NAMES:
    coordinates = spatial[name]
    neighbor_distance = NearestNeighbors(n_neighbors=NEIGHBORS + 1).fit(coordinates).kneighbors(coordinates)[0][:, -1]
    in_sparse_tissue  = neighbor_distance > np.quantile(neighbor_distance, 1 - SPARSE_FRACTION)
    counts = qc[name]['total_counts'].to_numpy()

    first_quiet_step_start[name] = np.nan
    for step_start in range(0, int(CANDIDATE_MIN_COUNTS.max()), STEP_WIDTH):
        in_step        = (counts >= step_start) & (counts < step_start + STEP_WIDTH)
        cells_in_step  = in_step.sum()
        lonely_in_step = (in_step & in_sparse_tissue).sum()
        p_value = binom.sf(lonely_in_step - 1, cells_in_step, SPARSE_FRACTION)
        if cells_in_step >= MIN_CELLS_PER_STEP and p_value >= ALPHA:
            first_quiet_step_start[name] = step_start
            break

# Adds cutoffs['enrichment_min_counts']: the first quiet step start, floored at the noise floor; in memory only.
# proposed_min_counts and every other column are not changed, so the filter step below is unaffected.
cutoffs['enrichment_min_counts'] = np.maximum(pd.Series(first_quiet_step_start), noise_min_counts)

cutoffs[['noise_min_counts', 'gmm_min_counts', 'enrichment_min_counts', 'proposed_min_counts']]

## Cutoffs on the distributions

Grey is the histogram of log10 values (cells with at least one count). Orange and blue are the low and high components of the fitted mixture.
Red dashed is the previous fixed cutoff, black is the mixture cut, green dotted is the noise floor,
purple dash-dot is the enrichment cut (counts only; absent when no quiet step was found).

In [ ]:
fig, axes = plt.subplots(len(SAMPLE_NAMES), 2, figsize=(14, 3.6 * len(SAMPLE_NAMES)), squeeze=False, gridspec_kw={'hspace': 0.5})
for row_axes, name in zip(axes, SAMPLE_NAMES):
    for ax, metric, previous, proposed in zip(
        row_axes,
        ['total_counts', 'n_genes'],
        [PREVIOUS_MIN_COUNTS, PREVIOUS_MIN_GENES],
        [cutoffs.loc[name, 'gmm_min_counts'], cutoffs.loc[name, 'gmm_min_genes']],
    ):
        values     = qc[name][metric].to_numpy()
        log_values = np.log10(values[values > 0])

        ax.hist(log_values, bins=80, color='lightgray')
        bin_width = (log_values.max() - log_values.min()) / 80
        x = np.linspace(log_values.min(), log_values.max(), 400)
        mixture = mixtures[(name, metric)]
        low = np.argmin(mixture.means_.ravel())
        for component, color, label in [(low, 'tab:orange', 'low component'), (1 - low, 'tab:blue', 'high component')]:
            mean     = mixture.means_[component, 0]
            sd       = np.sqrt(mixture.covariances_[component, 0, 0])
            expected = len(log_values) * bin_width * mixture.weights_[component] * norm.pdf(x, mean, sd)
            ax.plot(x, expected, color=color, lw=1.5, label=label)

        ax.axvline(np.log10(previous), color='tab:red', ls='--', label=f'previous ({previous})')
        ax.axvline(np.log10(proposed), color='black',            label=f'mixture cut ({proposed:.0f})')
        if metric == 'total_counts':
            ax.axvline(np.log10(noise_min_counts[name]), color='tab:green', ls=':', label=f'noise floor ({noise_min_counts[name]:.0f})')
            if not np.isnan(cutoffs.loc[name, 'enrichment_min_counts']):
                ax.axvline(np.log10(cutoffs.loc[name, 'enrichment_min_counts']), color='tab:purple', ls='-.', label=f"enrichment cut ({cutoffs.loc[name, 'enrichment_min_counts']:.0f})")

        ax.set_xlabel(f'log10({metric})')
        ax.set_title(f'{name} — {metric}')
        ax.legend(fontsize=8)

plt.show()

qc_label        = ln.ULabel(name='qc').save()
histogram_label = ln.ULabel(name='histogram').save()
cutoff_label    = ln.ULabel(name='cutoff').save()
save_fig(fig, 'enrichment_qc_gmm_cutoff_distributions', 'counts/genes histograms with Gaussian-mixture components, mixture cutoffs and enrichment cutoffs',
         ulabels=(qc_label, histogram_label, cutoff_label))

## Sweep: where do removed cells stop being special?

For each candidate minimum-count cutoff, take the cells it would remove and ask how many sit in sparse tissue
(distance to the 10th nearest cell in the top 10% for that sample). Enrichment of 1 means removed cells are
spread like any other cells; well above 1 means they concentrate in sparse regions, where you expect low-quality cells.
Orange is the fraction of cells removed. Red dashed is the previous cutoff, black is the mixture cutoff and
purple dash-dot is the enrichment cutoff.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, name in zip(axes.ravel(), SAMPLE_NAMES):
    coordinates = spatial[name]
    neighbor_distance = NearestNeighbors(n_neighbors=NEIGHBORS + 1).fit(coordinates).kneighbors(coordinates)[0][:, -1]
    in_sparse_tissue  = neighbor_distance > np.quantile(neighbor_distance, 1 - SPARSE_FRACTION)

    counts  = qc[name]['total_counts'].to_numpy()
    removed = counts[:, None] < CANDIDATE_MIN_COUNTS[None, :]
    removed_per_candidate = np.maximum(removed.sum(axis=0), 1)
    enrichment = (removed & in_sparse_tissue[:, None]).sum(axis=0) / removed_per_candidate / SPARSE_FRACTION

    ax.plot(CANDIDATE_MIN_COUNTS, enrichment, color='tab:blue')
    ax.axhline(1, color='gray', ls=':', label='no enrichment (1)')
    ax.set_xlabel('minimum counts')
    ax.set_ylabel('enrichment in sparse tissue (fold)', color='tab:blue')
    ax.set_title(name)

    fraction_axis = ax.twinx()
    fraction_axis.plot(CANDIDATE_MIN_COUNTS, removed.mean(axis=0), color='tab:orange')
    fraction_axis.set_ylabel('fraction of cells removed', color='tab:orange')

    ax.axvline(PREVIOUS_MIN_COUNTS, color='tab:red', ls='--', label=f'previous ({PREVIOUS_MIN_COUNTS})')
    ax.axvline(cutoffs.loc[name, 'proposed_min_counts'], color='black', label=f"mixture ({cutoffs.loc[name, 'proposed_min_counts']:.0f})")
    if not np.isnan(cutoffs.loc[name, 'enrichment_min_counts']):
        ax.axvline(cutoffs.loc[name, 'enrichment_min_counts'], color='tab:purple', ls='-.', label=f"enrichment ({cutoffs.loc[name, 'enrichment_min_counts']:.0f})")
    ax.legend(fontsize=8, loc='upper right')

fig.tight_layout()
plt.show()

sweep_label = ln.ULabel(name='sweep').save()
save_fig(fig, 'enrichment_qc_gmm_cutoff_sweep', 'enrichment of removed cells in sparse tissue across candidate count cutoffs, with mixture and enrichment cutoffs',
         ulabels=(qc_label, cutoff_label, sweep_label))

## Removed cells in space

Fraction of cells removed in each spatial bin. Left: previous fixed cutoffs. Middle: proposed (Gaussian mixture) counts and genes cutoffs.
Right: enrichment counts cutoff alone; it has no genes cutoff, so cells are removed on counts only and the fraction is not directly comparable with the other two.

In [ ]:
fig, axes = plt.subplots(len(SAMPLE_NAMES), 3, figsize=(18, 5 * len(SAMPLE_NAMES)), squeeze=False)
for row_axes, name in zip(axes, SAMPLE_NAMES):
    counts = qc[name]['total_counts']
    genes  = qc[name]['n_genes']
    removed_by_label = {
        'previous': (counts < PREVIOUS_MIN_COUNTS) | (genes < PREVIOUS_MIN_GENES),
        'proposed': (counts < cutoffs.loc[name, 'proposed_min_counts']) | (genes < cutoffs.loc[name, 'proposed_min_genes']),
        'enrichment (counts only)': counts < cutoffs.loc[name, 'enrichment_min_counts'],
    }
    for ax, (label, removed) in zip(row_axes, removed_by_label.items()):
        coordinates = spatial[name]
        bins = ax.hexbin(
            coordinates[:, 0], coordinates[:, 1], C=removed.to_numpy().astype(float),
            reduce_C_function=np.mean, gridsize=120, cmap='viridis', vmin=0, vmax=1, rasterized=True,
        )
        fig.colorbar(bins, ax=ax, shrink=0.6)
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_axis_off()
        has_cutoff = label != 'enrichment (counts only)' or not np.isnan(cutoffs.loc[name, 'enrichment_min_counts'])
        ax.set_title(f'{name} — {label}, {removed.mean():.1%} removed' + ('' if has_cutoff else ' (no cutoff found)'))

fig.tight_layout()
plt.show()

spatial_label = ln.ULabel(name='spatial').save()
save_fig(fig, 'enrichment_qc_gmm_cutoff_removed_cells_spatial', 'spatial fraction of cells removed, previous vs Gaussian-mixture vs enrichment cutoffs',
         ulabels=(qc_label, cutoff_label, spatial_label))

## Filter and save

Each sample is filtered at its own proposed cutoffs, then the samples are concatenated.
The per-sample cutoffs are stored in `uns['qc_cutoffs']` so they travel with the object.

In [ ]:
filtered = {}
for name in SAMPLE_NAMES:
    adata = samples[name]
    keep = (
        (adata.obs['total_counts'] >= cutoffs.loc[name, 'proposed_min_counts'])
        & (adata.obs['n_genes_by_counts'] >= cutoffs.loc[name, 'proposed_min_genes'])
    )
    # Drops cells below this sample's proposed counts or genes cutoff; kept cells are unchanged. In memory only.
    filtered[name] = adata[keep.to_numpy()].copy()

# Concatenates the filtered samples into one object and adds obs['sample']; in memory only.
combined = sc.concat(filtered, label='sample')

# Adds layers['counts']: a copy of the raw integer counts, since normalize_total overwrites .X.
combined.layers['counts'] = combined.X.copy()

# Adds uns['qc_cutoffs']: the per-sample cutoffs used above; saved to disk with the object below.
combined.uns['qc_cutoffs'] = cutoffs[['noise_min_counts', 'gmm_min_counts', 'gmm_min_genes', 'proposed_min_counts', 'proposed_min_genes']]

combined

In [ ]:
ln.Param(name='cutoff_method', dtype=str).save()
filtered_label = ln.ULabel(name='filtered').save()

# Writes a new lamindb artifact (key 'combined_filtered_gmm.h5ad'); the previous filtered artifact is not touched.
artifact = ln.Artifact.from_anndata(combined, key='combined_filtered.h5ad',
                                    description='Combined Xenium object after per-sample QC filtering with Gaussian-mixture cutoffs').save()
artifact.features.add_values({'cutoff_method': 'gaussian_mixture_per_sample', 'harmony': False})
artifact.ulabels.add(qc_label, cutoff_label, filtered_label)

ln.finish()